In [1]:
'''# How to run this code in Google Colab or Local Machine: 
#Before running this code, make sure to install the required python and needed packages. 
# Since Iam used to Visual studio code, below instructions are for VSC but can be true fot other similar IDEs.
# If you don't want to install them in Local Machine then you use the colab directly. 
# Choose Runtime -> Python 3(ipykernel)
# If the above is not available, Select another kernel -> Colab -> Auto Conect or New Colab server if you do not have

'''

import sys
from datetime import datetime
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Runtime is running Date and time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


''' Here we are using the Gdrive to store the file and read it from there. 
If you want to use the local machine or colab folder, you can use the commented code below to read the file.
'''
# If the file is stored in the local machine or in Colab folder, you can use the following code to read the file.
# file_path = r"C:\Users\veera\AI\SampleFile\test-product-review.parquet"
# file_path = r"\sample_data\test-product-review.parquet"

from google.colab import drive
import os
drive.mount('/content/drive')
print("Drive mounted:", os.path.ismount('/content/drive'))

file_path = '/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet'
print("File exists:"+file_path+" -", os.path.isfile(file_path))


# Read the Parquet file 
# with specific columns and filter for English language reviews
# For better understanding I have filtered the data for few columns English language reviews only. You can remove the filter if you want to read all the data.

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
                      filters=[[('language', '==', 'en')]])

#Print the first few rows of the DataFrame to see how the data looks like.
df = table.to_pandas()
df.info()
print(df.head(4))
#print(df.shape)
print("Count of each star rating:"+str(df["stars"].value_counts().to_dict()))

'''DATA CLEANING AND LABEL PREPARATION
Here we will convert the "stars" column to numeric and create a new "sentiment" column based on the star ratings.
We wil keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
We might also need to drop and empty or null values in the review_body"stars" column, as they cannot be used for sentiment analysis. Since Data is clean commenting it out. If you want to use it, you can uncomment the code below.
'''

# df = df[["review_body", "stars"]].dropna().copy()

# Convert stars to numeric as had problems with some non-numeric values in the dataset. Coerce errors to NaN, then drop those rows.
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
# df = df.dropna(subset=["stars"]).reset_index(drop=True)

# Keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["stars"].apply(create_sentiment_label)

# # Remove neutral examples for a simpler binary sentiment task
# df = df[df["sentiment"] != "neutral"].reset_index(drop=True)

print(df[["stars", "sentiment"]].head(6))
print(df["stars"].value_counts())
print(df["sentiment"].value_counts())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Executable: /usr/bin/python3
Runtime is running Date and time: 2026-10-30 21:10:27
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted: True
File exists:/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet - True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    460 non-null    object
 1   review_body  460 non-null    object
 2   language     460 non-null    object
 3   stars        460 non-null    int32 
dtypes: int32(1), object(3)
memory usage: 12.7+ KB
    review_id                                        review_body language  \
0  en_0254382  Well I haven't stunned anyone...yet. So I can ...       en   
1  en_0961337  I loved the first book so much had to read the...      

In [7]:
import warnings
import datasets
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

# --- SUPPRESS ALL EXCESS LOGS ---
warnings.filterwarnings("ignore")
datasets.disable_progress_bar()
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()

# 1. Dataset Setup (Assumes 'df' with 'review_body' and 'sentiment' columns is defined)
# df = pd.DataFrame(data)

# 2. Map string labels to integers
label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
inv_label_mapping = {0: "negative", 1: "neutral", 2: "positive"}
df["label"] = df["sentiment"].map(label_mapping)

# 3. Train/Test Split (Stratified)
train_df, test_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["label"]
)

# 4. Tokenization using AutoTokenizer (e.g., RoBERTa or BERT tokenizer)
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_len = 128


def tokenize_data(df_subset):
  encodings = tokenizer(
      df_subset["review_body"].tolist(),
      truncation=True,
      padding="max_length",
      max_length=max_len,
      return_tensors="pt",
  )
  labels = torch.tensor(df_subset["label"].values, dtype=torch.long)
  return encodings["input_ids"], labels


train_inputs, train_labels = tokenize_data(train_df)
test_inputs, test_labels = tokenize_data(test_df)


# 5. Custom PyTorch Dataset & DataLoader
class SentimentDataset(Dataset):

  def __init__(self, input_ids, labels):
    self.input_ids = input_ids
    self.labels = labels

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.labels[idx]


train_dataset = SentimentDataset(train_inputs, train_labels)
test_dataset = SentimentDataset(test_inputs, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# 6. Define Bidirectional LSTM Architecture
class SentimentLSTM(nn.Module):

  def __init__(
      self,
      vocab_size,
      embedding_dim,
      hidden_dim,
      output_dim,
      n_layers,
      dropout,
  ):
    super().__init__()
    self.embedding = nn.Embedding(
        vocab_size, embedding_dim, padding_idx=tokenizer.pad_token_id
    )
    self.lstm = nn.LSTM(
        embedding_dim,
        hidden_dim,
        num_layers=n_layers,
        bidirectional=True,
        dropout=dropout if n_layers > 1 else 0,
        batch_first=True,
    )
    self.fc = nn.Linear(hidden_dim * 2, output_dim)  # *2 for bidirectional
    self.dropout = nn.Dropout(dropout)

  def forward(self, text):
    # text shape: [batch_size, seq_len]
    embedded = self.dropout(self.embedding(text))

    # packed_output, (hidden, cell) = self.lstm(embedded)
    _, (hidden, _) = self.lstm(embedded)

    # Concatenate forward and backward final hidden states
    hidden = self.dropout(
        torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
    )

    return self.fc(hidden)


# Model Hyperparameters
VOCAB_SIZE = tokenizer.vocab_size
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
OUTPUT_DIM = 3
N_LAYERS = 2
DROPOUT = 0.3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentimentLSTM(
    VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

# 7. Training Loop
print("Starting LSTM Training...")
epochs = 4

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for batch_texts, batch_labels in train_loader:
    batch_texts, batch_labels = batch_texts.to(device), batch_labels.to(device)

    optimizer.zero_grad()
    predictions = model(batch_texts)
    loss = criterion(predictions, batch_labels)

    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  print(
      f"Epoch {epoch+1}/{epochs} | Training Loss:"
      f" {total_loss / len(train_loader):.4f}"
  )

# 8. Evaluation Loop on Test Set
model.eval()
correct_preds = 0
total_preds = 0

with torch.no_grad():
  for batch_texts, batch_labels in test_loader:
    batch_texts, batch_labels = batch_texts.to(device), batch_labels.to(device)
    outputs = model(batch_texts)
    preds = torch.argmax(outputs, dim=1)

    correct_preds += (preds == batch_labels).sum().item()
    total_preds += batch_labels.size(0)

accuracy = correct_preds / total_preds
print(f"\nEvaluating LSTM on Test Data:")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Starting LSTM Training...
Epoch 1/4 | Training Loss: 1.1013
Epoch 2/4 | Training Loss: 1.0971
Epoch 3/4 | Training Loss: 1.0945
Epoch 4/4 | Training Loss: 1.0894

Evaluating LSTM on Test Data:
Test Accuracy: 40.58%


In [3]:
import warnings
import datasets
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer

# --- SUPPRESS ALL EXCESS LOGS ---
warnings.filterwarnings("ignore")
datasets.disable_progress_bar()
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()

# 1. Dataset Setup (Assumes 'df' with 'review_body' and 'sentiment' columns is defined)
# df = pd.DataFrame(data)

# 2. Map string labels to integers
label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
inv_label_mapping = {0: "negative", 1: "neutral", 2: "positive"}
df["label"] = df["sentiment"].map(label_mapping)

# 3. Train/Test Split (Stratified)
train_df, test_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["label"]
)

# 4. Tokenization using AutoTokenizer
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_len = 128


def tokenize_data(df_subset):
  encodings = tokenizer(
      df_subset["review_body"].tolist(),
      truncation=True,
      padding="max_length",
      max_length=max_len,
      return_tensors="pt",
  )
  labels = torch.tensor(df_subset["label"].values, dtype=torch.long)
  return encodings["input_ids"], labels


train_inputs, train_labels = tokenize_data(train_df)
test_inputs, test_labels = tokenize_data(test_df)


# 5. Custom PyTorch Dataset & DataLoader
class SentimentDataset(Dataset):

  def __init__(self, input_ids, labels):
    self.input_ids = input_ids
    self.labels = labels

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.labels[idx]


train_dataset = SentimentDataset(train_inputs, train_labels)
test_dataset = SentimentDataset(test_inputs, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# 6. Define Bidirectional GRU Architecture
class SentimentGRU(nn.Module):

  def __init__(
      self,
      vocab_size,
      embedding_dim,
      hidden_dim,
      output_dim,
      n_layers,
      dropout,
  ):
    super().__init__()
    self.embedding = nn.Embedding(
        vocab_size, embedding_dim, padding_idx=tokenizer.pad_token_id
    )
    self.gru = nn.GRU(
        embedding_dim,
        hidden_dim,
        num_layers=n_layers,
        bidirectional=True,
        dropout=dropout if n_layers > 1 else 0,
        batch_first=True,
    )
    self.fc = nn.Linear(hidden_dim * 2, output_dim)  # *2 for bidirectional
    self.dropout = nn.Dropout(dropout)

  def forward(self, text):
    # text shape: [batch_size, seq_len]
    embedded = self.dropout(self.embedding(text))

    _, hidden = self.gru(embedded)

    # Concatenate forward and backward final hidden states from the last layer
    hidden = self.dropout(
        torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
    )

    return self.fc(hidden)


# Model Hyperparameters
VOCAB_SIZE = tokenizer.vocab_size
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
OUTPUT_DIM = 3
N_LAYERS = 2
DROPOUT = 0.3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentimentGRU(
    VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT
).to(device)

# Using Adam optimizer with a slight weight decay for regularization
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# 7. Training Loop
print("Starting Bidirectional GRU Training...")
epochs = 5

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for batch_texts, batch_labels in train_loader:
    batch_texts, batch_labels = batch_texts.to(device), batch_labels.to(device)

    optimizer.zero_grad()
    predictions = model(batch_texts)
    loss = criterion(predictions, batch_labels)

    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  print(
      f"Epoch {epoch+1}/{epochs} | Training Loss:"
      f" {total_loss / len(train_loader):.4f}"
  )

# 8. Evaluation Loop on Test Set
model.eval()
correct_preds = 0
total_preds = 0

with torch.no_grad():
  for batch_texts, batch_labels in test_loader:
    batch_texts, batch_labels = batch_texts.to(device), batch_labels.to(device)
    outputs = model(batch_texts)
    preds = torch.argmax(outputs, dim=1)

    correct_preds += (preds == batch_labels).sum().item()
    total_preds += batch_labels.size(0)

accuracy = correct_preds / total_preds
print(f"\nEvaluating GRU on Test Data:")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Starting Bidirectional GRU Training...
Epoch 1/5 | Training Loss: 1.0944
Epoch 2/5 | Training Loss: 0.9424
Epoch 3/5 | Training Loss: 0.8760
Epoch 4/5 | Training Loss: 0.7086
Epoch 5/5 | Training Loss: 0.6044

Evaluating GRU on Test Data:
Test Accuracy: 48.55%
